# Basic Transformation: Quantum Fourier Transform — CUDA-Q Notebook

**Chapter / concept:** Basic transformations — the **standard 3-qubit Quantum Fourier
Transform (QFT)**, including the conventional trailing SWAP network. Same transform,
same input, and the identical gate sequence as the Qiskit notebook.

**Transformation implemented:**
$$\mathrm{QFT}|x\rangle = \frac{1}{\sqrt{N}}\sum_{k=0}^{N-1} e^{2\pi i\, x k / N}\,|k\rangle,
\qquad N = 2^3 = 8$$

applied to the concrete computational-basis input **x = 5** (bitstring `101`).

### Global SETUP (identical in both notebooks)
- **N_QUBITS** = 3
- **Input state:** x = 5 → prepare `|101>` via `X` on q0 and q2 (q1 stays `|0>`)
- **Gate sequence (QFT proper, after input prep):**
  `H(q2) → CP(2π/4; q1→q2) → CP(2π/8; q0→q2) → H(q1) → CP(2π/4; q0→q1) → H(q0) → SWAP(q0,q2)`
- **QUBIT ORDERING CONTRACT:** qubit 0 = rightmost bit in every printed bitstring (Qiskit's
  native convention). **CUDA-Q's `get_state()` and `sample()` index qubit 0 as the
  most-significant / leftmost bit — the opposite of that contract.** This notebook
  explicitly reverses every bitstring it prints (amplitudes and counts) so its final
  dictionaries line up with the Qiskit notebook. See the "ordering fix" cell below.
- **SHOTS** = 10000, **SEED** = 42
- **GLOBAL PHASE CONVENTION:** normalize so the first amplitude with `abs(amp) > 1e-9`, in
  ascending bitstring order (after the bit-order fix), is rotated to be real and positive —
  the same normalization function used in the Qiskit notebook. This is automatically a
  no-op for any QFT output: the k=0 amplitude is always `1/sqrt(N)` (real, positive).
- **Basis ordering for matrices:** `{|000>, |001>, ..., |111>}` ascending binary order.

### Theory recap
Same as the Qiskit notebook: put `H` on a qubit, then a controlled-phase from every lower
qubit still to be processed with angle `2π/2^k`, then a trailing SWAP network to restore the
qubit-ordering convention. This exact gate sequence was independently verified against the
closed-form DFT matrix (for the full 8x8 unitary AND the specific `x=5` output amplitudes)
using a hand-written NumPy simulator (matching this notebook's bit-order/phase convention)
before being used here.

**Run all cells top to bottom.**

> **Note on this notebook's provenance:** CUDA-Q is not installable in the environment this
> notebook was authored in (no network access to PyPI/conda), so the CUDA-Q-specific API
> calls below could not be executed end-to-end here. The gate sequence, angles, and expected
> amplitudes *were* independently verified via NumPy (see the self-check note above and the
> diff-check cell at the end). Please run this notebook once in a CUDA-Q environment before
> trusting its printed numbers, and flag anything that doesn't match your installed version.


In [ ]:
# Install cell
# Package name: `cudaq` (NVIDIA CUDA-Q Python package).
# Platform caveat: CUDA-Q ships CPU and GPU-accelerated simulator backends; on machines
# without an NVIDIA GPU it will still run correctly on the CPU (`qpp-cpu`) backend, just slower.
#
# IMPORTANT: plain `!pip install ...` can silently install into a DIFFERENT Python
# environment than the one this notebook's kernel is actually running (common on
# Colab/Jupyter setups with more than one interpreter on PATH) -- that mismatch is what
# causes "ModuleNotFoundError: No module named 'cudaq'" on the next cell even though this
# install cell appears to run without errors. Using `{sys.executable} -m pip` pins the
# install to the exact interpreter this kernel uses.
import sys
!{sys.executable} -m pip install -q cudaq


In [ ]:
# Imports + constants
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import cudaq

N_QUBITS = 3
X_INPUT = 5                 # the computational-basis input |x=5> = |101>
SHOTS = 10000
SEED = 42
TOPIC_SLUG = "qft"

THETA_K2 = 2 * np.pi / (2**2)   # pi/2  -- identical value to the Qiskit notebook
THETA_K3 = 2 * np.pi / (2**3)   # pi/4  -- identical value to the Qiskit notebook

cudaq.set_random_seed(SEED)

print(f"N_QUBITS = {N_QUBITS}")
print(f"X_INPUT  = {X_INPUT}  (bitstring {format(X_INPUT, '03b')})")
print(f"SHOTS    = {SHOTS}")
print(f"SEED     = {SEED}")
print(f"THETA_K2 = {THETA_K2:.6f} rad (pi/2)")
print(f"THETA_K3 = {THETA_K3:.6f} rad (pi/4)")
print(f"CUDA-Q target: {cudaq.get_target().name}")


In [ ]:
# Build the kernel gate-by-gate, with a physical comment on each gate.
# NOTE: CUDA-Q's qubit 0 is the MOST-significant (leftmost) bit -- the opposite of the
# Qiskit contract -- which is why every readout below gets explicitly reversed later.
# The gate-by-gate structure and angles are identical to the Qiskit notebook.

@cudaq.kernel
def qft_kernel(theta_k2: float, theta_k3: float):
    q = cudaq.qvector(3)

    # --- input state prep: |x=5> = |101> (q2=1, q1=0, q0=1) ---
    x(q[0])
    x(q[2])

    # --- QFT proper ---
    # H on q2: starts mixing q2's phase information into an equal superposition.
    h(q[2])
    # Controlled phase from q1 (distance 1 -> k=2, angle 2pi/4): kicks a phase onto q2's
    # |1> component conditioned on q1.
    r1.ctrl(theta_k2, q[1], q[2])
    # Controlled phase from q0 (distance 2 -> k=3, angle 2pi/8): smaller phase kick since
    # q0 is "farther" from q2 in significance.
    r1.ctrl(theta_k3, q[0], q[2])
    # H on q1.
    h(q[1])
    # Controlled phase from q0 (distance 1 -> k=2, angle 2pi/4).
    r1.ctrl(theta_k2, q[0], q[1])
    # H on q0.
    h(q[0])
    # Final SWAP network: for n=3 there's a single swap, (q0, q2), which reverses the
    # qubit order so output index k uses the same qubit-ordering convention as input x.
    swap(q[0], q[2])

print(cudaq.draw(qft_kernel, THETA_K2, THETA_K3))


In [ ]:
# Draw the circuit: CUDA-Q has no native matplotlib circuit renderer, so after the
# text diagram above (ground truth), build a matplotlib figure styled to visually match
# the Qiskit circuit diagram (same wire layout, boxed gate labels, top-to-bottom qubit order).

def draw_matched_circuit(gate_ops, n_qubits, title, save_path):
    import matplotlib.patches as patches
    fig, ax = plt.subplots(figsize=(10, 2 + 0.6 * n_qubits))
    wire_y = {q: n_qubits - 1 - q for q in range(n_qubits)}
    n_steps = len(gate_ops)

    for q in range(n_qubits):
        ax.hlines(wire_y[q], 0, n_steps + 1, color="black", linewidth=1)
        ax.text(-0.3, wire_y[q], f"q{q}", ha="right", va="center", fontsize=11)

    for xi, g in enumerate(gate_ops, start=1):
        kind = g["kind"]
        if kind == "box":
            qb = g["qubit"]
            ax.add_patch(patches.Rectangle((xi - 0.25, wire_y[qb] - 0.25), 0.5, 0.5,
                                            facecolor="#6FA8DC", edgecolor="black", zorder=3))
            ax.text(xi, wire_y[qb], g["label"], ha="center", va="center", fontsize=10, zorder=4)
        elif kind == "control_box":
            c, t = g["control"], g["target"]
            ax.vlines(xi, wire_y[min(c, t)], wire_y[max(c, t)], color="black", zorder=2)
            ax.plot(xi, wire_y[c], "o", color="black", markersize=8, zorder=3)
            ax.add_patch(patches.Rectangle((xi - 0.3, wire_y[t] - 0.25), 0.6, 0.5,
                                            facecolor="#93C47D", edgecolor="black", zorder=3))
            ax.text(xi, wire_y[t], g["label"], ha="center", va="center", fontsize=9, zorder=4)
        elif kind == "swap":
            a, b = g["q1"], g["q2"]
            ax.vlines(xi, wire_y[min(a, b)], wire_y[max(a, b)], color="black", zorder=2)
            for qq in (a, b):
                y = wire_y[qq]
                ax.plot([xi - 0.15, xi + 0.15], [y - 0.15, y + 0.15], color="black", zorder=3)
                ax.plot([xi - 0.15, xi + 0.15], [y + 0.15, y - 0.15], color="black", zorder=3)

    ax.set_xlim(-1, n_steps + 1)
    ax.set_ylim(-1, n_qubits)
    ax.axis("off")
    ax.set_title(title)
    fig.tight_layout()
    fig.savefig(save_path, dpi=150, bbox_inches="tight")
    return fig

gate_ops = [
    {"kind": "box", "qubit": 0, "label": "X"},
    {"kind": "box", "qubit": 2, "label": "X"},
    {"kind": "box", "qubit": 2, "label": "H"},
    {"kind": "control_box", "control": 1, "target": 2, "label": "P(pi/2)"},
    {"kind": "control_box", "control": 0, "target": 2, "label": "P(pi/4)"},
    {"kind": "box", "qubit": 1, "label": "H"},
    {"kind": "control_box", "control": 0, "target": 1, "label": "P(pi/2)"},
    {"kind": "box", "qubit": 0, "label": "H"},
    {"kind": "swap", "q1": 0, "q2": 2},
]
fig = draw_matched_circuit(gate_ops, N_QUBITS, "QFT|x=5> circuit — CUDA-Q (matched to Qiskit style)",
                            f"{TOPIC_SLUG}_cudaq_circuit.png")
fig


In [ ]:
# Explicit matrix representation cell(s).
# CUDA-Q does not expose a general "extract the full unitary of this kernel" API the way
# qiskit.quantum_info.Operator does for a QuantumCircuit, so per rule 4 ("framework's own
# operator extraction where available") only the hand-built NumPy version is shown here --
# using the SAME basis ordering convention and the SAME construction as the Qiskit notebook,
# checked against the closed-form DFT matrix directly.

H_MAT = (1/np.sqrt(2)) * np.array([[1, 1], [1, -1]])
I_MAT = np.eye(2)

def phase_gate(theta):
    return np.array([[1, 0], [0, np.exp(1j * theta)]])

print("H matrix:")
print(np.round(H_MAT, 4))
print(f"\nPhase gate P(pi/2) matrix:")
print(np.round(phase_gate(THETA_K2), 4))
print(f"\nPhase gate P(pi/4) matrix:")
print(np.round(phase_gate(THETA_K3), 4))

def kron_all(mats):
    out = mats[0]
    for m in mats[1:]:
        out = np.kron(out, m)
    return out

def single(gate_2x2, qubit, n):
    mats = [gate_2x2 if pos == qubit else I_MAT for pos in range(n - 1, -1, -1)]
    return kron_all(mats)

def controlled_phase(theta, control, target, n):
    P1 = np.array([[0, 0], [0, 1]])
    P0 = np.array([[1, 0], [0, 0]])
    Pg = phase_gate(theta)
    mats_match, mats_other = [], []
    for pos in range(n - 1, -1, -1):
        if pos == control:
            mats_match.append(P1); mats_other.append(P0)
        elif pos == target:
            mats_match.append(Pg); mats_other.append(I_MAT)
        else:
            mats_match.append(I_MAT); mats_other.append(I_MAT)
    return kron_all(mats_match) + kron_all(mats_other)

def swap_op(q1, q2, n):
    dim = 2**n
    op = np.zeros((dim, dim))
    for idx in range(dim):
        bits = [(idx >> b) & 1 for b in range(n)]
        bits[q1], bits[q2] = bits[q2], bits[q1]
        new_idx = sum(bits[b] << b for b in range(n))
        op[new_idx, idx] = 1.0
    return op

hand_built_qft = single(H_MAT, 2, N_QUBITS)
hand_built_qft = controlled_phase(THETA_K2, 1, 2, N_QUBITS) @ hand_built_qft
hand_built_qft = controlled_phase(THETA_K3, 0, 2, N_QUBITS) @ hand_built_qft
hand_built_qft = single(H_MAT, 1, N_QUBITS) @ hand_built_qft
hand_built_qft = controlled_phase(THETA_K2, 0, 1, N_QUBITS) @ hand_built_qft
hand_built_qft = single(H_MAT, 0, N_QUBITS) @ hand_built_qft
hand_built_qft = swap_op(0, 2, N_QUBITS) @ hand_built_qft

print("\nHand-built QFT unitary (rows/cols in ascending |000>..|111> order, "
      "qubit-0-rightmost convention), rounded (first 4x4 corner shown for brevity):")
print(np.round(hand_built_qft[:4, :4], 4))

N = 2**N_QUBITS
dft_ref = np.zeros((N, N), dtype=complex)
for k in range(N):
    for x in range(N):
        dft_ref[k, x] = np.exp(2j * np.pi * x * k / N) / np.sqrt(N)
max_dft_diff = np.max(np.abs(hand_built_qft - dft_ref))
print(f"\nMax difference vs closed-form DFT matrix: {max_dft_diff:.2e}")
assert max_dft_diff < 1e-8, "Hand-built matrix does not match the standard DFT matrix!"
print("Cross-check PASSED: hand-built matrix implements the standard QFT/DFT matrix exactly.")
print("(This matrix is identical to the Qiskit notebook's hand-built matrix -- diff them "
      "directly if you want an extra cross-check.)")


In [ ]:
# Exact statevector cell: get_state(), reverse bit order to match the Qiskit contract,
# apply the same global-phase normalization function, print amplitudes + probability dict.

def normalize_global_phase(statevector, atol=1e-9):
    sv = np.array(statevector, dtype=complex).copy()
    for amp in sv:
        if abs(amp) > atol:
            phase = np.angle(amp)
            sv = sv * np.exp(-1j * phase)
            break
    return sv

raw_state = cudaq.get_state(qft_kernel, THETA_K2, THETA_K3)
raw_sv = np.array([complex(a) for a in raw_state])

n = N_QUBITS
# --- Bit-order fix (CUDA-Q only): CUDA-Q's index `idx` corresponds to bitstring
#     format(idx, '0{n}b') read with q0 as the LEFTMOST (most-significant) bit.
#     Reverse each bitstring so q0 becomes the rightmost bit, matching Qiskit.
reordered_sv = np.zeros_like(raw_sv)
for idx in range(2**n):
    cudaq_bitstring = format(idx, f"0{n}b")          # q0 = leftmost here
    fixed_bitstring = cudaq_bitstring[::-1]           # q0 = rightmost now (Qiskit contract)
    fixed_idx = int(fixed_bitstring, 2)
    reordered_sv[fixed_idx] = raw_sv[idx]

sv = normalize_global_phase(reordered_sv)

cudaq_prob_dict = {}
print("Statevector amplitudes (bitstring = q(n-1)...q0, qubit0 rightmost, AFTER reordering):")
for idx in range(2**n):
    bitstring = format(idx, f"0{n}b")
    amp = sv[idx]
    prob = abs(amp) ** 2
    cudaq_prob_dict[bitstring] = prob
    print(f"  |{bitstring}> (k={idx})  amp = {amp:.4f}   prob = {prob:.4f}")

cudaq_prob_dict = dict(sorted(cudaq_prob_dict.items()))
print("\nAll 8 probabilities should be equal (1/8), same as the Qiskit notebook.")


### Ordering-fix note (CUDA-Q)

**A bit reversal was required and applied above.** `cudaq.get_state()` returns amplitudes
indexed with **qubit 0 as the most-significant (leftmost) bit** — the opposite of the
Qiskit/reference convention (qubit 0 = rightmost bit) this whole exercise standardizes on.
The statevector cell above explicitly reversed every bitstring (`cudaq_bitstring[::-1]`) and
re-placed each amplitude at the corresponding reordered index, so the printed dictionary above
uses the same "qubit 0 = rightmost" convention as the Qiskit notebook, before either the phase
normalization or any downstream comparison happens. The measurement/counts cell below applies
the identical reversal to sampled bitstrings.

**Phase convention applied:** same function as the Qiskit notebook — rotate the whole vector
so the first amplitude with `abs(amp) > 1e-9`, in ascending bitstring order (evaluated *after*
the bit-order fix above), is real and positive. As in the Qiskit notebook, this is a no-op
for a QFT output.


In [ ]:
# Measurement cell: sample with the shared SEED/SHOTS, reorder bitstrings, get probabilities
cudaq.set_random_seed(SEED)
result = cudaq.sample(qft_kernel, THETA_K2, THETA_K3, shots_count=SHOTS)

raw_counts = {bitstring: count for bitstring, count in result.items()}

# Same bit-order fix as the statevector cell: reverse every sampled bitstring so qubit 0
# becomes the rightmost bit, matching the Qiskit contract.
fixed_counts = {}
for cudaq_bitstring, count in raw_counts.items():
    fixed_bitstring = cudaq_bitstring[::-1]
    fixed_counts[fixed_bitstring] = fixed_counts.get(fixed_bitstring, 0) + count

cudaq_sampled_probs = {bitstring: count / SHOTS for bitstring, count in fixed_counts.items()}
cudaq_sampled_probs = dict(sorted(cudaq_sampled_probs.items()))

print("Raw CUDA-Q counts (before bit-order fix):", dict(sorted(raw_counts.items())))
print("\nCounts after bit-order fix:", dict(sorted(fixed_counts.items())))
print("\nSampled probabilities (sorted by bitstring, post bit-order fix) -- measurement "
      "destroys phase information, so this should come out close to a flat 1/8:")
print({k: round(v, 4) for k, v in cudaq_sampled_probs.items()})


In [ ]:
# Histogram: matplotlib bar chart, saved to file, same axis range/labels/style as Qiskit's
fig2, ax2 = plt.subplots(figsize=(8, 4))
all_bitstrings = [format(i, f"0{N_QUBITS}b") for i in range(2**N_QUBITS)]
values = [cudaq_sampled_probs.get(b, 0.0) for b in all_bitstrings]
ax2.bar(all_bitstrings, values, color="#6FA8DC", edgecolor="black")
ax2.set_ylim(0, 0.3)
ax2.set_xlabel("bitstring (q2 q1 q0), i.e. output index k")
ax2.set_ylabel("probability")
ax2.set_title("QFT|x=5> sampled probabilities — CUDA-Q")
plt.xticks(rotation=45)
fig2.tight_layout()
fig2.savefig(f"{TOPIC_SLUG}_cudaq_histogram.png", dpi=150, bbox_inches="tight")
fig2


In [ ]:
# Resource counts.
# CUDA-Q kernels don't expose a public circuit-depth/gate-count API the way
# qiskit's QuantumCircuit.depth()/count_ops() do, so this is reported from the known
# kernel source directly: 2 X (input prep), 3 H, 3 controlled-phase (r1.ctrl), 1 SWAP.
gate_counts = {"x": 2, "h": 3, "r1_ctrl": 3, "swap": 1}
depth = 8
print(f"Circuit depth: {depth}")
print(f"Gate counts: {gate_counts}")


### Convention notes (restated)

- **Ordering fix applied:** yes — every bitstring from `cudaq.get_state()` and
  `cudaq.sample()` was reversed (`bitstring[::-1]`) so qubit 0 is the rightmost bit,
  matching the Qiskit notebook's native convention.
- **Phase convention applied:** global phase rotated so the first nonzero amplitude in
  ascending bitstring order (evaluated after the bit-order fix) is real and positive —
  a no-op here, as in the Qiskit notebook.


In [ ]:
# Final result dictionary
CUDAQ_RESULT = cudaq_sampled_probs
print("CUDAQ_RESULT =")
print({k: round(v, 4) for k, v in CUDAQ_RESULT.items()})


In [ ]:
# Diff-check block: hand-computed expected amplitudes/probabilities from the closed-form
# QFT formula, plus (commented) live comparison against the Qiskit notebook's result.

N = 2**N_QUBITS
HAND_COMPUTED_AMPLITUDES = {}
HAND_COMPUTED_EXPECTED = {}
for k in range(N):
    bitstring = format(k, f"0{N_QUBITS}b")
    amp = np.exp(2j * np.pi * X_INPUT * k / N) / np.sqrt(N)
    HAND_COMPUTED_AMPLITUDES[bitstring] = amp
    HAND_COMPUTED_EXPECTED[bitstring] = abs(amp) ** 2   # == 1/N for every k

print(f"Hand-computed expected amplitudes for QFT|x={X_INPUT}>:")
for k_str, amp in HAND_COMPUTED_AMPLITUDES.items():
    print(f"  |{k_str}>  amp={amp:.4f}")

print("\nHand-computed expected (ideal, noiseless) probabilities -- all equal to 1/8:")
print({k: round(v, 4) for k, v in HAND_COMPUTED_EXPECTED.items()})

max_delta_vs_ideal = max(abs(CUDAQ_RESULT.get(k, 0.0) - v) for k, v in HAND_COMPUTED_EXPECTED.items())
print(f"\nMax |sampled - ideal| delta (CUDA-Q, {SHOTS} shots): {max_delta_vs_ideal:.4f}")
# With finite shots this will be within normal statistical noise, not exactly 0.

amp_diff = np.max(np.abs(sv - np.array([HAND_COMPUTED_AMPLITUDES[format(k, f'0{N_QUBITS}b')] for k in range(N)])))
print(f"Max |statevector - formula| amplitude difference: {amp_diff:.2e}")
assert amp_diff < 1e-8, "Statevector amplitudes do not match the closed-form QFT formula!"
print("Cross-check PASSED: statevector amplitudes match the analytic QFT formula exactly.")

# QISKIT_RESULT = { ... }  # paste from the Qiskit notebook
# max_delta = max(abs(CUDAQ_RESULT[k] - QISKIT_RESULT[k]) for k in CUDAQ_RESULT)
# print(f"Max delta CUDAQ_RESULT vs QISKIT_RESULT: {max_delta:.4f}")
# assert max_delta < 1e-1, "Qiskit and CUDA-Q results diverge beyond shot noise!"
